# 01 — Exploratory Data Analysis

All data loaded via `pd.read_sql()` from SQLite. Charts are Plotly interactive.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from src.utils.db_connect import get_engine

engine = get_engine()
print('Connected to uniqlo.db')

## 1. Daily Sales Trend

In [ ]:
daily = pd.read_sql("""
    SELECT date, actual_sales, target, achievement_pct,
           gross_profit_amt, gross_profit_ratio,
           event_flag, temperature_index, num_customers
    FROM daily_sales ORDER BY date
""", engine, parse_dates=['date'])

# 7-day rolling average
daily['rolling_7d'] = daily['actual_sales'].rolling(7).mean()

fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    subplot_titles=['Daily Sales vs Target', 'Gross Profit Ratio'],
                    row_heights=[0.7, 0.3])

fig.add_trace(go.Scatter(x=daily['date'], y=daily['actual_sales'],
                          mode='lines', name='Actual Sales',
                          line=dict(color='#1f77b4', width=1), opacity=0.6), row=1, col=1)
fig.add_trace(go.Scatter(x=daily['date'], y=daily['target'],
                          mode='lines', name='Target',
                          line=dict(color='#ff7f0e', dash='dash', width=1.5)), row=1, col=1)
fig.add_trace(go.Scatter(x=daily['date'], y=daily['rolling_7d'],
                          mode='lines', name='7-day Rolling Avg',
                          line=dict(color='#2ca02c', width=2.5)), row=1, col=1)

fig.add_trace(go.Scatter(x=daily['date'], y=daily['gross_profit_ratio'],
                          mode='lines', name='GP Ratio',
                          line=dict(color='#9467bd', width=1)), row=2, col=1)

fig.update_layout(title='Daily Sales 2024–2025', height=600, hovermode='x unified')
fig.update_yaxes(title_text='AUD', tickformat='$,.0f', row=1)
fig.update_yaxes(title_text='GP %', tickformat='.1%', row=2)
fig.show()

## 2. Day-of-Week Sales Distribution

In [ ]:
dow_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
daily['day_of_week'] = pd.Categorical(daily['day_of_week'], categories=dow_order, ordered=True)

fig = px.box(daily, x='day_of_week', y='actual_sales',
             title='Sales Distribution by Day of Week',
             labels={'actual_sales': 'Daily Sales (AUD)', 'day_of_week': ''},
             color='day_of_week', color_discrete_sequence=px.colors.qualitative.Set2)
fig.update_yaxes(tickformat='$,.0f')
fig.update_layout(showlegend=False)
fig.show()

## 3. Division Revenue Share

In [ ]:
div_annual = pd.read_sql("""
    SELECT division_code, division_name, department,
           SUM(sales_amt) AS total_sales,
           AVG(gp_ratio) AS avg_gp
    FROM division_daily
    GROUP BY division_code, division_name, department
    ORDER BY total_sales DESC
""", engine)

fig = px.bar(div_annual, x='total_sales', y='division_name',
             orientation='h', color='department',
             title='Total Revenue by Division (2024–2025)',
             labels={'total_sales': 'Total Sales (AUD)', 'division_name': ''},
             color_discrete_map={'Women': '#e6b3c8', 'Men': '#a8c8e8',
                                  'Kids': '#c8e8a8', 'Others': '#e8dfa8'})
fig.update_xaxes(tickformat='$,.0f')
fig.update_layout(height=550)
fig.show()

## 4. Seasonal Demand Heatmap

In [ ]:
monthly_div = pd.read_sql("""
    SELECT CAST(strftime('%m', date) AS INTEGER) AS month_num,
           division_name,
           ROUND(AVG(sales_amt), 0) AS avg_daily_sales
    FROM division_daily
    GROUP BY month_num, division_name
""", engine)

pivot = monthly_div.pivot(index='division_name', columns='month_num', values='avg_daily_sales')
month_labels = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

fig = px.imshow(pivot, labels=dict(x='Month', y='Division', color='Avg Daily Sales (AUD)'),
                x=month_labels, color_continuous_scale='YlOrRd',
                title='Avg Daily Sales Heatmap by Month × Division')
fig.update_layout(height=600)
fig.show()

## 5. Temperature vs Outerwear Sales

In [ ]:
outerwear = pd.read_sql("""
    SELECT ds.date, ds.temperature_index, ds.event_flag,
           SUM(CASE WHEN dd.division_code IN (21,31) THEN dd.sales_amt ELSE 0 END) AS outerwear_sales,
           SUM(CASE WHEN dd.division_code IN (24,34) THEN dd.sales_amt ELSE 0 END) AS cutsewn_sales
    FROM daily_sales ds
    JOIN division_daily dd ON ds.date = dd.date
    GROUP BY ds.date
""", engine, parse_dates=['date'])

fig = px.scatter(outerwear, x='temperature_index', y='outerwear_sales',
                 color='event_flag', opacity=0.6,
                 title='Temperature vs Outerwear Sales (Div 21 + 31)',
                 labels={'temperature_index': 'Temperature (°C)',
                          'outerwear_sales': 'Outerwear Sales (AUD)'},
                 trendline='lowess')
fig.update_yaxes(tickformat='$,.0f')
fig.show()

## 6. GP% by Division

In [ ]:
gp_div = pd.read_sql("""
    SELECT division_name, department,
           ROUND(AVG(gp_ratio) * 100, 2) AS avg_gp_pct,
           ROUND(SUM(sales_amt), 0) AS total_sales
    FROM division_daily
    GROUP BY division_code, division_name, department
    ORDER BY avg_gp_pct DESC
""", engine)

fig = px.bar(gp_div, x='avg_gp_pct', y='division_name', orientation='h',
             color='department', title='Average GP% by Division',
             labels={'avg_gp_pct': 'Avg GP%', 'division_name': ''},
             color_discrete_map={'Women': '#e6b3c8', 'Men': '#a8c8e8',
                                  'Kids': '#c8e8a8', 'Others': '#e8dfa8'})
fig.update_xaxes(ticksuffix='%')
fig.update_layout(height=550)
fig.show()

## 7. YoY Growth by Month

In [ ]:
yoy = pd.read_sql("""
    SELECT strftime('%m', date) AS month_num,
           SUM(CASE WHEN strftime('%Y', date)='2024' THEN actual_sales ELSE 0 END) AS sales_2024,
           SUM(CASE WHEN strftime('%Y', date)='2025' THEN actual_sales ELSE 0 END) AS sales_2025
    FROM daily_sales
    GROUP BY month_num ORDER BY month_num
""", engine)
yoy['yoy_pct'] = (yoy['sales_2025'] - yoy['sales_2024']) / yoy['sales_2024'] * 100
month_labels = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
yoy['month'] = month_labels

fig = px.bar(yoy, x='month', y='yoy_pct',
             title='YoY Revenue Growth by Month (2024 → 2025)',
             labels={'yoy_pct': 'YoY Change %', 'month': ''},
             color='yoy_pct', color_continuous_scale='RdYlGn',
             color_continuous_midpoint=0)
fig.update_yaxes(ticksuffix='%')
fig.show()

## 8. Customer Segment Spend Comparison

In [ ]:
seg = pd.read_sql("""
    SELECT customer_type,
           COUNT(*) AS num_customers,
           ROUND(AVG(avg_spend_per_visit), 2) AS avg_spend,
           ROUND(AVG(total_visits), 1) AS avg_visits,
           ROUND(SUM(total_spend), 0) AS total_revenue
    FROM customers
    GROUP BY customer_type
""", engine)

fig = make_subplots(rows=1, cols=2,
                    subplot_titles=['Avg Spend per Visit by Segment',
                                    'Total Revenue Contribution'])
colors = ['#2196F3', '#4CAF50', '#FF9800']
fig.add_trace(go.Bar(x=seg['customer_type'], y=seg['avg_spend'],
                      marker_color=colors, name='Avg Spend'), row=1, col=1)
fig.add_trace(go.Bar(x=seg['customer_type'], y=seg['total_revenue'],
                      marker_color=colors, name='Total Revenue', showlegend=False), row=1, col=2)
fig.update_yaxes(tickformat='$,.0f', row=1, col=1)
fig.update_yaxes(tickformat='$,.0f', row=1, col=2)
fig.update_layout(title='Customer Segment Analysis', showlegend=False)
fig.show()